# IFEval on ScreamingFace: stable first run, then the research experiment

IFEval (arXiv:2311.07911) is 541 prompts with machine-checkable constraints — word
counts, forbidden punctuation, required sections. The Engine grades every response with
a deterministic verifier: **no judge model in the grading path, zero grading cost**.

The Engine publishes three independently revisioned IFEval Benchmarks that share Cases and verifier
assets:

- `ifeval` — one shot. A solo Model answers once; a Fusion's members answer and its
  synthesizer **blends** them into one new answer. The blend is checked.
- `ifeval/self-corrective` — three fixed attempts. The whole Candidate reads the
  checker's violations, **writes its own feedback, and retries**.
- `ifeval/verifying-ensemble` — the current verifying ensemble implementation based on
  Skurikhin et al. (https://openreview.net/forum?id=XSIYfTm2h7): every direct Fusion
  member is checked individually, and the **synthesizer acts as JUDGE** — it picks a
  passing answer word-for-word, or turns the violations into coaching when nobody
  passed. It never writes the answer on this exam.

One rule to remember: **the synthesizer plays two roles.** Blender on `ifeval`,
judge on `ifeval/verifying-ensemble`.

The required cells below use Haiku and Gemini Flash for a provider-stable one-Case
validation. Khoa's Kimi K3 configuration remains in the optional appendix because a
reasoning model can consume its completion budget before emitting answer text, and an
upstream provider can fail even when Gateway discovery succeeds.

## Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105 and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

After pulling or merging SDK code, **restart this notebook's kernel before Run
All**. Python keeps already-imported SDK modules in memory; a stale kernel can ask the new Engine
for a pre-merge Benchmark id and receive `unknown_benchmark`.

In [ ]:
import screamingface as sf

In [ ]:
sf.connect()

## Stable smoke Candidates

These four cells are a **paid one-Case validation**, not a scientific result. Haiku is the
solo Candidate. The Fusion pairs Haiku with Gemini Flash and uses Flash as its synthesizer,
so the synthesizer is also a direct member. This is a cheap product smoke configuration,
not one of Skurikhin et al.'s published model lineups.

`progress=True` shows the live Engine stream. Raw URL4 node names are expected until
semantic Case/attempt events land.

Execution is disabled by default. Set `RUN_EVALUATION = True` deliberately; **Run All** otherwise
makes no model calls.

In [ ]:
smoke_model = sf.Model(
    "openrouter/anthropic/claude-haiku-4.5",
    params={"max_tokens": 4096},
)
smoke_judge = sf.Model(
    "openrouter/google/gemini-3-flash-preview",
    params={"max_tokens": 4096},
)

smoke_fusion = sf.Fusion(
    [smoke_model, smoke_judge],
    name="haiku-flash-smoke",
    synthesizer="openrouter/google/gemini-3-flash-preview",
    params={"max_tokens": 4096},
)
smoke_fusion

In [ ]:
RUN_EVALUATION = False

## ① Baseline — one model, one shot

This validates the canonical one-answer plus deterministic-check execution shape. One Case
is only a plumbing check; run all 541 official Case keys for a benchmark score.

In [ ]:
canonical_smoke_model = (
    sf.evaluate(
        smoke_model,
        benchmark="ifeval",
        limit=1,
        progress=True,
    )
    if RUN_EVALUATION
    else None
)
canonical_smoke_model or "Evaluation disabled — set RUN_EVALUATION = True to spend."

## ② Does blending preserve instructions?

The synthesizer writes one NEW answer from the members' answers — new text the checker
never saw. A blend can break a constraint every member satisfied (add a comma, drop a
section). This cell measures that risk.

In [ ]:
canonical_smoke_fusion = (
    sf.evaluate(
        smoke_fusion,
        benchmark="ifeval",
        limit=1,
        progress=True,
    )
    if RUN_EVALUATION
    else None
)
canonical_smoke_fusion or "Evaluation disabled — set RUN_EVALUATION = True to spend."

## ③ Can a model correct itself?

The ablation the paper never ran: {solo + feedback loop}. The model answers, the
checker reports violations, the model writes its own feedback and retries. The current
Variant always executes exactly three attempts; the earliest pass wins.

Cost: five model calls per case (three answers + two self-feedback authorings), all
unrolled.

In [ ]:
corrective_smoke_model = (
    sf.evaluate(
        smoke_model,
        benchmark="ifeval/self-corrective",
        limit=1,
        progress=True,
    )
    if RUN_EVALUATION
    else None
)
corrective_smoke_model or "Evaluation disabled — set RUN_EVALUATION = True to spend."

## ④ ScreamingFace verifying-ensemble variant

Members answer, the checker checks **each draft individually**, and the synthesizer —
acting as judge here — picks a passing answer verbatim, or coaches everyone and retries
when nobody passed. A judge cannot break a constraint a member satisfied, because it
never rewrites the winning text.

This Variant is inspired by Skurikhin et al., but it is not their exact protocol: it
executes all three rounds and Judge steps unconditionally, supports two to four members,
and uses locally defined Judge/retry prompts.

Choose a synthesizer that reliably answers tersely: a judge reply that is not a bare
letter gets no vote (the deterministic passers-first rule decides instead), and the
synthesizer inherits provider-default params on this exam.

In [ ]:
corrective_smoke_fusion = (
    sf.evaluate(
        smoke_fusion,
        benchmark="ifeval/verifying-ensemble",
        limit=1,
        progress=True,
    )
    if RUN_EVALUATION
    else None
)
corrective_smoke_fusion or "Evaluation disabled — set RUN_EVALUATION = True to spend."

## Read the smoke results

- ① vs ② — did blending help or hurt instruction-following?
- ① vs ③ — how much does a feedback loop help one model?
- ③ vs ④ — self-correction vs ensemble correction, same loop, same exam.
- ② vs ④ — blend-then-check vs check-then-select.

Cost note: the iterative-correction exam has no early stop yet — all three attempts
always run (and the solo shape adds two self-feedback calls), so its token totals
overstate a stop-on-success system. Compare scores freely within a column; never
compare our costs to the paper's.

With one Case these values prove only that the complete contracts execute. They are not
benchmark results.

In [ ]:
{
    name: {
        "score": report.candidates[0].score,
        "output_tokens": report.usage.output_tokens,
    }
    for name, report in {
        "① ifeval · haiku": canonical_smoke_model,
        "② ifeval · haiku-flash": canonical_smoke_fusion,
        "③ self-corrective · haiku": corrective_smoke_model,
        "④ verifying-ensemble · haiku-flash": corrective_smoke_fusion,
    }.items()
    if report is not None
}

## Optional appendix — Khoa's Kimi K3 experiment

This preserves the original research lineup without making it the environment-health
check. It is disabled so **Run All does not spend on it or fail because of an upstream K3
completion**.

Observed failure meanings:

- `case … carried no valid IFEval check record` means a Candidate/provider failed to return
  a scorable answer. The Engine refuses to turn that into a plausible zero score.
- `aigateway returned neither answer content nor tool calls` commonly means a reasoning
  model exhausted its completion budget before emitting final answer text.
- An HTTP 200 from the upstream call does not prove answer content was present.

Set `RUN_KIMI_RESEARCH = True` only after the smoke grid succeeds. Start with one Case;
increase `KIMI_RESEARCH_LIMIT` deliberately. Kimi K3 may need a larger completion budget,
but a higher ceiling cannot repair an upstream provider error.

In [ ]:
RUN_KIMI_RESEARCH = False
KIMI_RESEARCH_LIMIT = 1

kimi = sf.Model("openrouter/moonshotai/kimi-k3", params={"max_tokens": 4096})
haiku = sf.Model("openrouter/anthropic/claude-haiku-4.5")
kimi_fusion = sf.Fusion(
    [kimi, haiku],
    name="kimi-haiku",
    synthesizer="openrouter/moonshotai/kimi-k3",
    params={"max_tokens": 4096},
)

"Enabled" if RUN_KIMI_RESEARCH else "Skipped — set RUN_KIMI_RESEARCH = True to opt in"

In [ ]:
if RUN_KIMI_RESEARCH:
    kimi_canonical = sf.evaluate(
        kimi,
        benchmark="ifeval",
        limit=KIMI_RESEARCH_LIMIT,
        progress=True,
    )
    kimi_blended = sf.evaluate(
        kimi_fusion,
        benchmark="ifeval",
        limit=KIMI_RESEARCH_LIMIT,
        progress=True,
    )
    kimi_self_corrective = sf.evaluate(
        kimi,
        benchmark="ifeval/self-corrective",
        limit=KIMI_RESEARCH_LIMIT,
        progress=True,
    )
    kimi_verifying_ensemble = sf.evaluate(
        kimi_fusion,
        benchmark="ifeval/verifying-ensemble",
        limit=KIMI_RESEARCH_LIMIT,
        progress=True,
    )
    kimi_results = {
        "① ifeval · kimi": kimi_canonical,
        "② ifeval · kimi-haiku": kimi_blended,
        "③ self-corrective · kimi": kimi_self_corrective,
        "④ verifying-ensemble · kimi-haiku": kimi_verifying_ensemble,
    }
else:
    kimi_results = {}

{
    name: {
        "score": report.candidates[0].score,
        "output_tokens": report.usage.output_tokens,
    }
    for name, report in kimi_results.items()
}